# Concept Drift & Adaptation

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/streaming-ml/04-concept-drift

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

**The problem.** On a live stream the data-generating distribution changes over time (users, fraud, seasons), so a fixed model rots. **The core idea:** watch the model's error stream with a **change detector**; when it fires, **adapt** (retrain / reset). We build a simplified **ADWIN** from scratch and cross-check it against the `river` library's `drift.ADWIN`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## 1. From scratch — a stream with a sudden concept shift

Before `D` a model is accurate (error ~0.08); after `D` the concept changes and a **static** model degrades (error ~0.42). The error bits are what a detector actually sees.

In [ ]:
N, D = 2000, 1000
err_good, err_bad = 0.08, 0.42
p = np.where(np.arange(N) < D, err_good, err_bad)
static_err = (np.random.rand(N) < p).astype(int)   # 1 = misclassified
print('mean error before D:', round(static_err[:D].mean(), 3),
      '| after D:', round(static_err[D:].mean(), 3))

### A simplified ADWIN detector

Keep a window of recent error bits. If some split into an older half `W0` and a newer half `W1` has means differing by more than the Hoeffding cut `ε_cut = √( (1/2m)·ln(4|W|/δ) )` (with `m` the harmonic mean of the sub-window sizes), declare drift at that point.

In [ ]:
def adwin_detect(bits, delta=0.05, min_sub=30):
    start = 0
    for t in range(min_sub, len(bits)):
        win = bits[start:t + 1]
        L = len(win)
        pre = np.cumsum(win)
        for s in range(min_sub, L - min_sub):
            n0, n1 = s, L - s
            m0 = pre[s - 1] / n0
            m1 = (pre[-1] - pre[s - 1]) / n1
            m = 1 / (1 / n0 + 1 / n1)
            e_cut = np.sqrt((1 / (2 * m)) * np.log(4 * L / delta))
            if abs(m0 - m1) > e_cut:
                return t     # detection time
    return None

det = adwin_detect(static_err, delta=0.05)
print(f'true drift at D={D}, detected at t={det}  (delay {det - D})')
assert det is not None and D <= det <= D + 200, 'should detect shortly after the true drift'

## 2. The library way — validate against river's ADWIN

`river` is the standard online-ML library; `drift.ADWIN` is the production detector. We feed it the *same* error stream and check it also flags drift soon after `D`. (Install if needed — falls back gracefully offline.)

In [ ]:
try:
    from river import drift
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'river'], check=False)
    try:
        from river import drift
    except ImportError:
        drift = None

if drift is not None:
    adwin = drift.ADWIN(delta=0.002)
    river_det = None
    for i, b in enumerate(static_err):
        adwin.update(int(b))
        if adwin.drift_detected:
            river_det = i
            break
    print(f'river ADWIN detected drift at t={river_det}')
    assert river_det is not None and D <= river_det <= D + 300, 'river should detect after D too'
    print('our from-scratch ADWIN and river.drift.ADWIN both fire shortly after the shift ✓')
else:
    print('river unavailable offline — from-scratch detector already validated above')

## 3. Visualize it — detect, then adapt

An **adaptive** model retrains at detection, so its error falls back to baseline; the **static** model stays degraded.

In [ ]:
adaptive_err = static_err.copy()
adaptive_err[det:] = (np.random.rand(N - det) < err_good).astype(int)  # retrained at detection
def smooth(b, w=40):
    return np.convolve(b, np.ones(w) / w, mode='valid')
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(smooth(static_err), color='#f43f5e', label='static (no adapt)')
ax.plot(smooth(adaptive_err), color='#14b8a6', label='adaptive (ADWIN)')
ax.axvline(D, ls='--', color='#eab308', label='true drift')
ax.axvline(det, ls=':', color='#14b8a6', label='detected')
ax.set_ylabel('error rate'); ax.set_xlabel('stream position')
ax.set_title('Detect, then adapt', color='white')
ax.legend(); ax.grid(alpha=0.2); plt.show()

**What to notice:** both models spike when the concept shifts, but the adaptive one recovers to the baseline error a short **detection delay** after `D`, while the static model stays stuck at the degraded rate. The gap between the curves is the cost of *not* adapting.

## 4. Tradeoffs & when to use it

- **Drift type matters.** Covariate shift (`P(X)` changes, `P(Y|X)` fixed) may need no action; real drift (`P(Y|X)` changes) invalidates the boundary and demands adaptation.
- **δ is the core knob.** Larger `δ` → smaller `ε_cut` → faster detection but more false alarms; smaller `δ` is conservative.
- **Adaptation strategies.** Windowed retrain, incremental update, detect-and-reset, or ensembles (recurring concepts favour ensembles).
- **Hoeffding trees** stream decision-tree induction: split once the best attribute beats the runner-up by more than the Hoeffding bound `ε = √(R²ln(1/δ)/2n)`.

## 5. Your turn

### Exercise 1 — DDM (Drift Detection Method)

DDM tracks the online error rate `p_t` and its std `s_t = √(p_t(1-p_t)/t)`, remembering the minimum of `p_t + s_t`. It flags **drift** when `p_t + s_t ≥ p_min + 3·s_min`. Return the first index where that happens (start monitoring after `warmup` samples).

In [ ]:
def ddm_detect(bits, warmup=30):
    p_min, s_min = np.inf, np.inf
    n_err, seen = 0, 0
    for i, b in enumerate(bits):
        seen += 1; n_err += int(b)
        p = n_err / seen
        s = np.sqrt(p * (1 - p) / seen)
        if seen < warmup:
            continue
        # TODO(you): update p_min/s_min when (p + s) is a new minimum,
        #            then return i if p + s >= p_min + 3*s_min
        ...
    return None


In [ ]:
# Checks — run me
np.random.seed(0)
stream = np.concatenate([
    (np.random.rand(1000) < 0.05).astype(int),   # low-error regime
    (np.random.rand(1000) < 0.40).astype(int),   # error jumps -> drift
])
d = ddm_detect(stream)
assert d is not None and 1000 <= d <= 1300, f'DDM should flag soon after index 1000, got {d}'
assert ddm_detect((np.random.rand(1500) < 0.05).astype(int)) is None, 'no drift on a stable stream'
print('✅ Exercise 1 passed  (drift flagged at index %d)' % d)

<details>
<summary>💡 Show solution</summary>

```python
def ddm_detect(bits, warmup=30):
    p_min, s_min = np.inf, np.inf
    n_err, seen = 0, 0
    for i, b in enumerate(bits):
        seen += 1; n_err += int(b)
        p = n_err / seen
        s = np.sqrt(p * (1 - p) / seen)
        if seen < warmup:
            continue
        if p + s < p_min + s_min:
            p_min, s_min = p, s
        if p + s >= p_min + 3 * s_min:
            return i
    return None
```

</details>

## 6. Key takeaways

- **Concept drift** is change in `P_t(X, Y)`; separate covariate shift from dangerous real drift.
- **ADWIN** cuts its window when two halves diverge by more than a Hoeffding cut; **DDM** watches `2σ/3σ` error bands.
- The detector's `δ` trades detection delay against false alarms.
- Next: [Streaming ML in Production](https://ml-viz-ruby.vercel.app/courses/streaming-ml/05-streaming-ml-in-production).